In [10]:
# Librerias
import pandas as pd
import numpy as np
import random

In [11]:
# Carga de los datos

RUTA_CR = "../datos/CR.txt"

COLUMNAS = [
    "geonameid", "nombre", "nombre_ascii", "nombres_alternos", "latitud", "longitud",
    "clase", "codigo", "pais", "cc2", "admin1", "admin2", "admin3", "admin4",
    "poblacion", "elevacion", "dem", "zona_horaria", "fecha_modificacion",
]

def leer_geonames(ruta=RUTA_CR):
    df = pd.read_csv(
        ruta,
        sep="\t",
        header=None,
        names=COLUMNAS,
        dtype={"admin1": str, "admin2": str, "admin3": str, "admin4": str, "cc2": str},
    )
    return df

df = leer_geonames()

In [12]:
# filtrado
pob_min = 1
N_LOCALIDADES = 15

df_filtrado = df[df["clase"] == "P"]
df_filtrado = df_filtrado[~df_filtrado["codigo"].isin(["PPLQ", "PPLW", "PPLH"])]
df_filtrado = df_filtrado[df_filtrado["poblacion"] >= pob_min]

# ordenar primero (desempate por geonameid) para conservar, entre repetidos, el más poblado
df_filtrado = df_filtrado.sort_values(["poblacion", "geonameid"], ascending=[False, True])

# coordenadas repetidas = iguales al redondear a 3 decimales (~100 m)
df_filtrado = df_filtrado[~df_filtrado[["latitud", "longitud"]].round(3).duplicated()]

df_filtrado = df_filtrado.head(N_LOCALIDADES).reset_index(drop=True)
df_filtrado.head()

,geonameid,nombre,nombre_ascii,nombres_alternos,latitud,longitud,clase,codigo,pais,cc2,admin1,admin2,admin3,admin4,poblacion,elevacion,dem,zona_horaria,fecha_modificacion
0,3621849,San José,San Jose,"SJO,San Chose,San Chosė,San Hose,San Jose,San ...",9.93388,-84.08489,P,PPLC,CR,NaN,08,101,10103,NaN,335007,NaN,1133,America/Costa_Rica,2025-05-15
1,3622247,Limón,Limon,"Ciudad de Limon,Ciudad de Limón,LIO,Limon,Limo...",9.99074,-83.03596,P,PPLA,CR,NaN,06,701,70101,NaN,63081,NaN,29,America/Costa_Rica,2026-04-13
2,3621911,San Francisco,San Francisco,San Francisco,9.99380,-84.12742,P,PPLX,CR,NaN,04,401,40103,NaN,55923,NaN,1131,America/Costa_Rica,2021-11-11
3,3624955,Alajuela,Alajuela,"Alachouela,Alachuela,Alajuela,Alakhuehla,Alaxu...",10.01723,-84.21275,P,PPLA,CR,NaN,01,201,20101,NaN,47494,NaN,955,America/Costa_Rica,2026-04-13
4,3623076,Liberia,Liberia,"LIR,Liberia,Liberii,Liberija,Liberiya,Liberya,...",10.63517,-85.43779,P,PPLA,CR,NaN,03,501,50101,NaN,45380,NaN,148,America/Costa_Rica,2026-04-13


In [13]:
# Análisis descriptivo de los datos

# estructura del archivo completo
df.info()
display(df["clase"].value_counts())

# resumen de las localidades seleccionadas
display(df_filtrado[["poblacion", "latitud", "longitud"]].describe())
display(df_filtrado["codigo"].value_counts())
display(df_filtrado["admin1"].value_counts())

# cadena de filtrado y proporción de población cero
print("Registros totales / lugares poblados / seleccionadas:",
      len(df), len(df[df["clase"] == "P"]), len(df_filtrado))
print("Proporción de lugares poblados con población 0:",
      round((df[df["clase"] == "P"]["poblacion"] == 0).mean(), 3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21426 entries, 0 to 21425
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   geonameid           21426 non-null  int64  
 1   nombre              21426 non-null  object 
 2   nombre_ascii        21426 non-null  object 
 3   nombres_alternos    18538 non-null  object 
 4   latitud             21426 non-null  float64
 5   longitud            21426 non-null  float64
 6   clase               21426 non-null  object 
 7   codigo              21426 non-null  object 
 8   pais                21426 non-null  object 
 9   cc2                 71 non-null     object 
 10  admin1              21364 non-null  object 
 11  admin2              5079 non-null   object 
 12  admin3              4923 non-null   object 
 13  admin4              1 non-null      object 
 14  poblacion           21426 non-null  int64  
 15  elevacion           77 non-null     float64
 16  dem 

clase
H    6801
P    5900
S    4401
T    2908
L     798
A     565
V      44
R       6
U       3
Name: count, dtype: int64

,poblacion,latitud,longitud
count,15.000000,15.000000,15.000000
mean,58191.333333,10.016097,-84.156583
std,77283.551671,0.356474,0.592221
min,27477.000000,9.367400,-85.437790
25%,30732.000000,9.908805,-84.170085
50%,34877.000000,9.956890,-84.050740
75%,46437.000000,9.992270,-83.949320
max,335007.000000,10.951920,-83.035960


codigo
PPL      7
PPLA     4
PPLC     1
PPLX     1
PPLA3    1
PPLA2    1
Name: count, dtype: int64

admin1
08    7
01    2
02    2
06    1
04    1
03    1
07    1
Name: count, dtype: int64

Registros totales / lugares poblados / seleccionadas: 21426 5900 15
Proporción de lugares poblados con población 0: 0.978


In [14]:
# Coordenadas planas (km) y matriz de distancias euclidianas

R_TIERRA = 6371.0  # radio medio de la Tierra (km)

lat = np.radians(df_filtrado["latitud"].to_numpy())
lon = np.radians(df_filtrado["longitud"].to_numpy())
lat0, lon0 = lat.mean(), lon.mean()  # punto de referencia: centro del conjunto

# Proyección equirectangular
x = R_TIERRA * np.cos(lat0) * (lon - lon0)
y = R_TIERRA * (lat - lat0)
coords = np.column_stack((x, y))
df_filtrado["x_km"] = x
df_filtrado["y_km"] = y

# Matriz de distancias euclidianas (n x n)
diferencias = coords[:, None, :] - coords[None, :, :]
matriz_distancias = np.sqrt((diferencias ** 2).sum(axis=2))
n = len(df_filtrado)

In [15]:
# Tabla de localidades seleccionadas y exportación para MATLAB

PROVINCIAS = {"01": "Alajuela", "02": "Cartago", "03": "Guanacaste", "04": "Heredia",
              "06": "Limón", "07": "Puntarenas", "08": "San José"}

tabla = pd.DataFrame({
    "Índice": df_filtrado.index,
    "Nombre": df_filtrado["nombre"],
    "Provincia": df_filtrado["admin1"].map(PROVINCIAS),
    "Código": df_filtrado["codigo"],
    "Población": df_filtrado["poblacion"],
    "Latitud": df_filtrado["latitud"].round(3),
    "Longitud": df_filtrado["longitud"].round(3),
    "x (km)": df_filtrado["x_km"].round(1),
    "y (km)": df_filtrado["y_km"].round(1),
})

# Tabla en formato Markdown
print(tabla.to_markdown(index=False).replace(".", ","))

# Exportar para MATLAB
df_filtrado[["x_km", "y_km", "poblacion"]].to_csv("../datos/localidades_seleccionadas.csv", index_label="indice")

|   Índice | Nombre        | Provincia   | Código   |   Población |   Latitud |   Longitud |   x (km) |   y (km) |
|---------:|:--------------|:------------|:---------|------------:|----------:|-----------:|---------:|---------:|
|        0 | San José      | San José    | PPLC     |      335007 |     9,934 |    -84,085 |      7,9 |     -9,1 |
|        1 | Limón         | Limón       | PPLA     |       63081 |     9,991 |    -83,036 |    122,7 |     -2,8 |
|        2 | San Francisco | Heredia     | PPLX     |       55923 |     9,994 |    -84,127 |      3,2 |     -2,5 |
|        3 | Alajuela      | Alajuela    | PPLA     |       47494 |    10,017 |    -84,213 |     -6,2 |      0,1 |
|        4 | Liberia       | Guanacaste  | PPLA     |       45380 |    10,635 |    -85,438 |   -140,3 |     68,8 |
|        5 | Paraíso       | Cartago     | PPL      |       39702 |     9,838 |    -83,867 |     31,8 |    -19,8 |
|        6 | Puntarenas    | Puntarenas  | PPLA     |       35650 |     9,977 | 

## Función para el cálculo de la distancia entre rutas para los algoritmos

In [16]:
def longitud_ruta(ruta, matriz_distancias):
    """
    Funcion para calcular la longitud total de una ruta cerrada.

    Entradas:                 ruta ---- Lista con los indices de las ciudades
                 matriz_distancias ---- Matriz de distancias de tamanyo nxn

    Salidas:              longitud ---- Longitud total, incluido el regreso
                                        a la ciudad inicial
    """

    n = len(ruta) # Cantidad de localidades
    longitud = 0 # Contador de la longitud total

    for i in range(n):
        if i == n - 1:
            j = 0  # La ultima ciudad se conecta con la primera, supuesto del problema
        else:
            j = i + 1

        a = ruta[i]
        b = ruta[j]

        longitud = longitud + matriz_distancias[a, b]

    return longitud

# 2-opt

# Y-opt

El algoritmo Y-opt es una variante del algoritmo 2-opt. Como se vio anteriormente, el algoritmo 2-opt acepta un intercambio de aristas si la suma de las longitudes de las dos aristas nuevas es menor que la suma de las dos aristas viejas; esto se hace comparando únicamente las aristas que se modificaron, ya que el resto de la ruta permanece igual. Por otro lado, el algoritmo Y-opt impone una condición de cambio más estricta, donde, para aceptar un cambio en la ruta, la arista más larga de las nuevas debe ser más corta que la arista más larga de las viejas, y, además, la arista más corta de las nuevas debe ser más corta que la arista más corta de las viejas. Esto hace que el algoritmo Y-opt sea más restrictivo en su condición de aceptación de nuevas rutas, ya que no solo compara la suma total, sino el tamaño individual de cada arista.

La principal ventaja del algoritmo Y-opt es que el número de iteraciones para una ruta de $n$ vértices y $m$ aristas está acotado por $(m-1)n$ (Manthey & van Rhijn, 2025), resultando en un tiempo de ejecución polinomial. En comparación, el algoritmo 2-opt puede necesitar un tiempo exponencial en los peores casos; de esta forma, el algoritmo Y-opt compensa su condición de aceptación con el tiempo que necesitará para ejecutarse. A pesar de que el algoritmo Y-opt sea más estricto, los resultados teóricos muestran que este algoritmo tiene resultados similares a los del 2-opt, donde se llega a rutas ligeramente más largas con el algoritmo Y-opt pero en un tiempo de ejecución acotado en comparación al 2-opt (Manthey & van Rhijn, 2025).

In [17]:
# Funcion Y-opt

def y_opt(ruta_inicial, matriz_distancias):
    """
    Función para implementar el algoritmo Y-opt
    Entradas :           ruta_inicial ---- Lista con los indices de las ciudades 
                    matriz_distancias ---- Matriz de distancias de tamanyo nxn
                    
    Salidas :             ruta_nueva ---- Lista con la nueva ruta optimizada
    """
    
    n = len(ruta_inicial)
    ruta_copia = ruta_inicial.copy() # Copy para evitar sobreescribir la lista original
    mejorada = True 
    
    while mejorada == True: # Hacer el while hasta que la ruta no se pueda mejorar mas
        mejorada = False # Cambiar la condicion de si se mejoro o no la ruta (Se va a cambiar más adelante)
        
        for i in range(n-1):
            for j in range(i + 2, n): # i + 2 para evitar usar la misma arista en las comparaciones
                if (j == n-1): # Se llega a la última ciudad, la cual debe conectar con la primera siempre
                    k = 0
                else:
                    k = j + 1 # Si no estamos en la última ciudad, la ciudad actual se conecta con la siguiente
               
                # Las 4 ciudades consecutivas son: (Ordenadas segun la ruta inicial)
                a = ruta_copia[i]
                b = ruta_copia[i+1]
                c = ruta_copia[j]
                d = ruta_copia[k]
                
                # Largo de las aristas viejas:
                dist_vieja1 = matriz_distancias[a,b]
                dist_vieja2 = matriz_distancias[c,d]
                
                # Largo de las aristas nuevas:
                dist_nueva1 = matriz_distancias[a,c]
                dist_nueva2 = matriz_distancias[b,d]
                
                # Crear lista con las distancias nuevas
                
                dist_viejas = np.array([dist_vieja1,dist_vieja2])
                dist_nuevas = np.array([dist_nueva1,dist_nueva2])
                
                if (np.max(dist_nuevas) < np.max(dist_viejas)) and (np.min(dist_nuevas) < np.min(dist_viejas)): # Si esto se cumple se puede cambiar la ruta
                    ruta_nueva = ruta_copia[:i+1] + ruta_copia[i+1:j+1][::-1] + ruta_copia[j+1:] # Los + son para concatenar las listas, hay que recordar que en Python las listas empiezan en 0 y se incluye el extremo izquierdo pero no el derecho cuando se extrae una sublista; el [::-1] es para darle vuelta al segmento que está entre las aristas cambiadas
                    ruta_copia = ruta_nueva
                    mejorada = True
                    break # Salir del for de j
            if mejorada == True:
                break  # Salir del for de i
          
    return ruta_copia
            
            
    

In [18]:
# Método para ejecutar los tres algoritmos

def ejecutar_algoritmos(matriz_distancias, n):
    """
    Metodo para ejecutar los algoritmos 2-opt, X-opt y Y-opt
    Entradas:         matriz_distancias  ---- Matriz de tamanyo nxn que posee las distancias entre las ciudades
                                      n  ---- Numero de ciudades
    Salidas:                ruta_inicial ---- Ruta inicial creada aleatoriamente
                              ruta_2_opt ---- Ruta optimizada con el algoritmo de 2-opt
                              ruta_X_opt ---- Ruta optimizada con el algoritmo de X-opt
                              ruta_Y_opt ---- Ruta optimizada con el algoritmo de Y-opt                       
    """
    ruta_inicial = list(range(n)) # Crear una lista de n elementos
    random.shuffle(ruta_inicial) # Ordenar la lista de la ruta de forma aleatoria 
    
    # Aplicar los tres algoritmos
    ruta_dos_opt = dos_opt(ruta_inicial, matriz_distancias)
    ruta_x_opt = x_opt(ruta_inicial, matriz_distancias)
    ruta_y_opt = y_opt(ruta_inicial, matriz_distancias)
    
    return ruta_inicial, ruta_dos_opt, ruta_x_opt, ruta_y_opt
    


# Medidas a Recopilar en los experimentos

Para evaluar y poder comparar el rendimiento de los tres algoritmos, 2-opt, X-opt y Y-opt, se utilizarán las siguientes medidas:

Longitud de la ruta optimizada: Es la longitud final medida en kilómetros de cada ruta optimizada según el algoritmo; esta medida incluye todo el recorrido realizado y el regreso desde la última visitada hasta el punto de partida inicial. De esta forma, se puede comparar qué tan grande o pequeña es la ruta de un algoritmo con respecto a las demás. 

Tiempo de ejecución: Es el tiempo medido en segundos, de lo que tardó cada algoritmo en encontrar su solución. Esto permitirá comparar la rapidez de cada algoritmo para encontrar una solución. 

Número de intercambios aceptados: Es el número de veces que cada algoritmo aceptó algún cambio en la ruta. Esta medida permitirá ver qué tanto aceptó o rechazó cambios un algoritmo, permitiendo así, ver qué tan estricto es cada algoritmo con la aceptación de un cambio. 

Ruta final obtenida: Es una lista los índices que representan el orden en que se visitó cada ciudad. Esto permitirá realizar los gráficos y comparar las rutas creadas por cada algoritmo.

Porcentaje de mejora con respecto a la ruta inicial: Esta medida se calculará como $$((longitud inicial - longitud final) / longitud inicial) \times 100 $$; esta medida permite ver cuánto mejoró la solución de cada algoritmo con respecto a la ruta inicial creada aleatoriamente.

Las métricas antes mencionadas permitirán realizar la comparación entre algoritmos, además de la visualización de cada ruta creada. Es importante destacar que, para lograr realizar esta comparación, los tres algoritmos utilizarán la misma ruta inicial y la misma matriz de distancias.